# 🎨 LoRA Training — Alice (SDXL) в Google Colab

**Тренировка LoRA на 15 фото Алисы**

Использует [ai-toolkit](https://github.com/ostris/ai-toolkit) от Ostris.

**GPU:** T4 (бесплатно) | **Время:** ~30-60 мин | **Результат:** `.safetensors` LoRA

In [ ]:
#@title ⚙️ Шаг 1: Установка (запустить один раз)

# Клонируем наш репозиторий с датасетом
!git clone https://github.com/Aurora-VP/alice-lora-training.git /content/alice-lora-data

# Клонируем ai-toolkit
!git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
%cd /content/ai-toolkit

# Устанавливаем зависимости
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q xformers

import os
DATASET_DIR = '/content/alice-lora-data/dataset'
OUTPUT_DIR = '/content/alice-lora-output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Проверяем датасет
pngs = [f for f in os.listdir(DATASET_DIR) if f.endswith('.png')]
print(f'✅ Датасет: {len(pngs)} изображений')
print(f'✅ ai-toolkit установлен')
print(f'OUTPUT: {OUTPUT_DIR}')

In [ ]:
#@title 🎯 Шаг 2: Конфигурация

TRIGGER_WORD = "alice"  #@param {type:"string"}
NUM_EPOCHS = 15  #@param {type:"slider", min:5, max:30, step:1}
LEARNING_RATE = 0.0001  #@param {type:"number"}
RANK = 16  #@param [8, 16, 32, 64] {type:"raw"}
RESOLUTION = 1024  #@param [512, 768, 1024] {type:"raw"}

config = {
  "model": {
    "name_or_path": "stabilityai/stable-diffusion-xl-base-1.0",
    "is_sdxl": True
  },
  "datasets": [{
    "folder_path": DATASET_DIR,
    "caption_ext": "txt",
    "default_caption": f"{TRIGGER_WORD} woman portrait, detailed face, high quality",
    "resolution": RESOLUTION,
    "repeats": 10,
    "crop_jitter": 20,
    "crop_style": "center",
    "crop_aspect": "square"
  }],
  "train": {
    "batch_size": 1,
    "steps": NUM_EPOCHS * 150,
    "lr": LEARNING_RATE,
    "noise_scheduler": "ddpm",
    "train_unet": True,
    "train_text_encoder": False,
    "adam_weight_decay": 0.01,
    "gradient_checkpointing": True,
    "mixed_precision": "fp16"
  },
  "save": {
    "dtype": "float16",
    "save_every": 500,
    "max_step_saves_to_keep": 5,
    "save_dir": OUTPUT_DIR
  },
  "lora": {
    "rank": RANK,
    "alpha": RANK,
    "dropout": 0.0
  },
  "sample": {
    "sample_every": 250,
    "prompts": [
      f"{TRIGGER_WORD} woman portrait, professional photo, studio lighting",
      f"{TRIGGER_WORD} woman, outdoor, natural lighting, detailed face",
      f"{TRIGGER_WORD} woman, close-up portrait, soft lighting"
    ],
    "negative_prompt": "blurry, low quality, distorted, deformed",
    "seed": 42,
    "walk_seed": True
  }
}

import json
config_path = '/content/alice_lora_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'✅ Конфигурация сохранена')
print(f'   Trigger: {TRIGGER_WORD}')
print(f'   Epochs: {NUM_EPOCHS}, Steps: {NUM_EPOCHS * 150}')
print(f'   Rank: {RANK}, Resolution: {RESOLUTION}')

In [ ]:
#@title 🚀 Шаг 3: Тренировать LoRA!
%cd /content/ai-toolkit
print('🚀 Запуск тренировки... (~30-60 мин на T4)')
print('========================================')
!python run.py /content/alice_lora_config.json

print('')
print('✅ Тренировка завершена!')
print(f'LoRA сохранена в: {OUTPUT_DIR}')
print('Файлы:')
!ls -la {OUTPUT_DIR}/*.safetensors 2>/dev/null || echo 'Пока нет .safetensors файлов'

In [ ]:
#@title 📥 Шаг 4: Скачать результат
import glob, os
from google.colab import files

lora_files = glob.glob(f'{OUTPUT_DIR}/**/*.safetensors', recursive=True)
if lora_files:
    latest = max(lora_files, key=os.path.getmtime)
    print(f'📦 Скачиваю: {latest}')
    files.download(latest)
else:
    print('❌ LoRA файлы не найдены. Запустите тренировку.')